In [1]:
"""
generate_performance_data.py
-----------------------------
Generates realistic mock budget vs actual performance data
with embedded variances for diagnostic testing.
"""

import pandas as pd
import numpy as np
import os

np.random.seed(42)

CATEGORIES = ["Skincare", "Haircare", "Bath & Body", "Oral Care"]
MONTHS = pd.date_range("2023-01-01", periods=12, freq="MS").strftime("%Y-%m").tolist()

def generate():
    os.makedirs("data", exist_ok=True)
    os.makedirs("output/charts", exist_ok=True)

    budget_records = []
    actual_records = []

    for month in MONTHS:
        for cat in CATEGORIES:
            # Budget — steady growth assumption
            bud_volume = np.random.randint(800, 1500)
            bud_price  = round(np.random.uniform(180, 450), 2)
            budget_records.append({
                "month": month, "category": cat,
                "budget_volume": bud_volume,
                "budget_price":  bud_price,
                "budget_revenue": round(bud_volume * bud_price, 2),
            })

            # Actual — add realistic variance
            # Introduce a bad quarter (Q3: Jul-Sep) for Haircare
            if cat == "Haircare" and month in ["2023-07","2023-08","2023-09"]:
                vol_factor   = np.random.uniform(0.60, 0.75)  # volume shortfall
                price_factor = np.random.uniform(0.90, 0.98)  # slight price pressure
            elif cat == "Skincare" and month in ["2023-10","2023-11","2023-12"]:
                vol_factor   = np.random.uniform(1.10, 1.25)  # festive season uplift
                price_factor = np.random.uniform(1.02, 1.08)
            else:
                vol_factor   = np.random.uniform(0.88, 1.12)
                price_factor = np.random.uniform(0.95, 1.05)

            act_volume = int(bud_volume * vol_factor)
            act_price  = round(bud_price * price_factor, 2)
            actual_records.append({
                "month": month, "category": cat,
                "actual_volume":  act_volume,
                "actual_price":   act_price,
                "actual_revenue": round(act_volume * act_price, 2),
            })

    pd.DataFrame(budget_records).to_csv("data/budget_targets.csv",    index=False)
    pd.DataFrame(actual_records).to_csv("data/actual_performance.csv", index=False)

    print("Generated:")
    print("  data/budget_targets.csv")
    print("  data/actual_performance.csv")
    print(f"  {len(budget_records)} budget records / {len(actual_records)} actual records")


if __name__ == "__main__":
    generate()

Generated:
  data/budget_targets.csv
  data/actual_performance.csv
  48 budget records / 48 actual records


In [2]:
"""
diagnostic_framework.py
------------------------
Root Cause Analysis Framework for Financial Variances

Decomposes budget vs actual variance into:
  - Volume variance   : caused by selling more/fewer units than planned
  - Price variance    : caused by achieving higher/lower price than planned
  - Mix contribution  : driven by category mix shifts

Produces waterfall chart, trend chart, and narrative root cause report.


"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
from datetime import datetime

BUDGET_FILE  = "data/budget_targets.csv"
ACTUAL_FILE  = "data/actual_performance.csv"
OUT_VAR_CSV  = "output/variance_decomposition.csv"
OUT_REPORT   = "output/root_cause_summary.txt"
CHART_DIR    = "output/charts"

SIGNIFICANT_THRESHOLD = 0.10   # flag variances > 10% from budget

findings = []


# ── Load & Merge ─────────────────────────────────────────────────────────────

def load_data():
    print("\n" + "="*60)
    print("STEP 1: LOAD DATA")
    print("="*60)

    bud = pd.read_csv(BUDGET_FILE)
    act = pd.read_csv(ACTUAL_FILE)

    df = pd.merge(bud, act, on=["month","category"], how="outer")
    df = df.fillna(0)
    print(f"  Merged: {len(df)} rows | {df['month'].nunique()} months | {df['category'].nunique()} categories")
    return df


# ── Price-Volume Decomposition ───────────────────────────────────────────────

def decompose_variance(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standard Price-Volume-Mix decomposition:
      Volume variance = (Actual Vol - Budget Vol) × Budget Price
      Price variance  = (Actual Price - Budget Price) × Actual Vol
      Total variance  = Actual Revenue - Budget Revenue

    Assumption: Mix variance is the residual. Avoids double-counting.
    """
    print("\n" + "="*60)
    print("STEP 2: PRICE-VOLUME DECOMPOSITION")
    print("="*60)

    df["volume_variance"] = (
        (df["actual_volume"] - df["budget_volume"]) * df["budget_price"]
    ).round(2)

    df["price_variance"] = (
        (df["actual_price"] - df["budget_price"]) * df["actual_volume"]
    ).round(2)

    df["total_variance"] = (df["actual_revenue"] - df["budget_revenue"]).round(2)

    df["mix_variance"] = (
        df["total_variance"] - df["volume_variance"] - df["price_variance"]
    ).round(2)

    df["variance_pct"] = (
        np.where(
            df["budget_revenue"] != 0,
            (df["total_variance"] / df["budget_revenue"] * 100).round(1),
            0
        )
    )

    df["is_significant"] = df["variance_pct"].abs() > (SIGNIFICANT_THRESHOLD * 100)

    # Summary
    total_bud = df.groupby("month")["budget_revenue"].sum()
    total_act = df.groupby("month")["actual_revenue"].sum()
    total_var = (total_act - total_bud).sum()

    print(f"\n  Total Budget Revenue : ₹{total_bud.sum():,.0f}")
    print(f"  Total Actual Revenue : ₹{total_act.sum():,.0f}")
    print(f"  Total Variance       : ₹{total_var:,.0f} ({round(total_var/total_bud.sum()*100,1)}%)")

    findings.append(f"Total Budget Revenue  : ₹{total_bud.sum():,.0f}")
    findings.append(f"Total Actual Revenue  : ₹{total_act.sum():,.0f}")
    findings.append(f"Overall Variance      : ₹{total_var:,.0f} ({round(total_var/total_bud.sum()*100,1)}%)")

    return df


# ── Identify Root Causes ─────────────────────────────────────────────────────

def identify_root_causes(df: pd.DataFrame):
    """
    Rule-based root cause identification.
    Flags months/categories with significant variance and diagnoses driver.
    """
    print("\n" + "="*60)
    print("STEP 3: ROOT CAUSE IDENTIFICATION")
    print("="*60)

    flagged = df[df["is_significant"]].copy()
    print(f"\n  Significant variances found: {len(flagged)}")

    causes = []
    for _, row in flagged.iterrows():
        drivers = []
        if abs(row["volume_variance"]) > abs(row["price_variance"]):
            primary = "VOLUME"
            direction = "shortfall" if row["volume_variance"] < 0 else "uplift"
            drivers.append(f"Primary driver: {primary} {direction} "
                           f"(₹{row['volume_variance']:,.0f})")
        else:
            primary = "PRICE"
            direction = "pressure" if row["price_variance"] < 0 else "uplift"
            drivers.append(f"Primary driver: {primary} {direction} "
                           f"(₹{row['price_variance']:,.0f})")

        if abs(row["mix_variance"]) > 0.2 * abs(row["total_variance"]):
            drivers.append(f"Mix shift also contributing "
                           f"(₹{row['mix_variance']:,.0f})")

        cause_str = " | ".join(drivers)
        causes.append({
            "month": row["month"],
            "category": row["category"],
            "total_variance": row["total_variance"],
            "variance_pct": row["variance_pct"],
            "root_cause": cause_str,
        })
        print(f"  [{row['month']}] {row['category']:15} "
              f"₹{row['total_variance']:>10,.0f} ({row['variance_pct']:>+.1f}%) — {cause_str}")

    findings.append(f"\n--- Root Causes ---")
    for c in causes:
        findings.append(
            f"  [{c['month']}] {c['category']}: "
            f"₹{c['total_variance']:,.0f} ({c['variance_pct']:+.1f}%) | {c['root_cause']}"
        )

    return pd.DataFrame(causes)


# ── Visualisations ────────────────────────────────────────────────────────────

def visualise(df: pd.DataFrame):
    print("\n" + "="*60)
    print("STEP 4: GENERATING CHARTS")
    print("="*60)

    os.makedirs(CHART_DIR, exist_ok=True)
    plt.rcParams.update({"font.family": "DejaVu Sans", "figure.dpi": 120})

    # ── Chart 1: Actual vs Budget Revenue Trend ──────────────────────────
    monthly = df.groupby("month")[["budget_revenue","actual_revenue"]].sum().reset_index()

    fig, ax = plt.subplots(figsize=(13, 5))
    x = range(len(monthly))
    ax.plot(x, monthly["budget_revenue"]/1e6, "o--", color="#ADB5BD",
            linewidth=2, label="Budget", markersize=6)
    ax.plot(x, monthly["actual_revenue"]/1e6, "o-", color="#2E75B6",
            linewidth=2.5, label="Actual", markersize=7)
    ax.fill_between(x,
                    monthly["budget_revenue"]/1e6,
                    monthly["actual_revenue"]/1e6,
                    alpha=0.12,
                    color=np.where(
                        (monthly["actual_revenue"] >= monthly["budget_revenue"]).values,
                        "#1D9E75", "#E03131"
                    )[0])
    ax.set_xticks(list(x))
    ax.set_xticklabels(monthly["month"], rotation=45, ha="right", fontsize=9)
    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda v, _: f"₹{v:.1f}M")
    )
    ax.set_title("Monthly Revenue: Actual vs Budget", fontsize=14, pad=12)
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"{CHART_DIR}/trend_vs_budget.png")
    plt.close()
    print("  Saved trend_vs_budget.png")

    # ── Chart 2: Category Contribution to Total Variance ─────────────────
    cat_var = df.groupby("category")["total_variance"].sum().sort_values()

    colors = ["#E03131" if v < 0 else "#1D9E75" for v in cat_var.values]
    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.barh(cat_var.index, cat_var.values/1e6, color=colors, alpha=0.85)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title("Category Contribution to Total Variance (₹M)", fontsize=14, pad=12)
    ax.set_xlabel("Variance (₹M)")
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"₹{v:.1f}M"))
    for bar in bars:
        w = bar.get_width()
        ax.text(w + (0.02 if w >= 0 else -0.02),
                bar.get_y() + bar.get_height()/2,
                f"₹{w:.2f}M", va="center",
                ha="left" if w >= 0 else "right", fontsize=9)
    plt.tight_layout()
    plt.savefig(f"{CHART_DIR}/category_contribution.png")
    plt.close()
    print("  Saved category_contribution.png")

    # ── Chart 3: Waterfall — Volume vs Price vs Mix ───────────────────────
    vol_total = df["volume_variance"].sum()
    prc_total = df["price_variance"].sum()
    mix_total = df["mix_variance"].sum()
    total_var = df["total_variance"].sum()

    labels  = ["Budget\nRevenue", "Volume\nVariance", "Price\nVariance",
               "Mix\nVariance", "Actual\nRevenue"]
    bud_rev = df["budget_revenue"].sum()
    act_rev = df["actual_revenue"].sum()
    values  = [bud_rev, vol_total, prc_total, mix_total, act_rev]

    running = [0, bud_rev, bud_rev + vol_total,
               bud_rev + vol_total + prc_total,
               0]  # actual starts from 0

    bar_colors = [
        "#2E75B6",
        "#1D9E75" if vol_total >= 0 else "#E03131",
        "#1D9E75" if prc_total >= 0 else "#E03131",
        "#1D9E75" if mix_total >= 0 else "#E03131",
        "#2E75B6",
    ]

    fig, ax = plt.subplots(figsize=(10, 6))
    for i, (label, value, bottom, color) in enumerate(
        zip(labels, values, running, bar_colors)
    ):
        if i in (0, 4):
            ax.bar(i, value/1e6, color=color, alpha=0.85, width=0.6)
        else:
            ax.bar(i, value/1e6, bottom=bottom/1e6, color=color, alpha=0.85, width=0.6)

        ax.text(i, (bottom + (value if i not in (0,4) else 0))/1e6 +
                (0.3 if value >= 0 else -0.8),
                f"₹{value/1e6:.1f}M",
                ha="center", va="bottom" if value >= 0 else "top", fontsize=9)

    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, fontsize=10)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"₹{v:.0f}M"))
    ax.set_title("Variance Waterfall: Volume vs Price vs Mix", fontsize=14, pad=12)
    ax.axhline(0, color="black", linewidth=0.6)

    pos_patch = mpatches.Patch(color="#1D9E75", label="Favourable")
    neg_patch = mpatches.Patch(color="#E03131", label="Adverse")
    base_patch = mpatches.Patch(color="#2E75B6", label="Budget / Actual")
    ax.legend(handles=[pos_patch, neg_patch, base_patch], loc="lower right")

    plt.tight_layout()
    plt.savefig(f"{CHART_DIR}/waterfall_variance.png")
    plt.close()
    print("  Saved waterfall_variance.png")


# ── Save Outputs ─────────────────────────────────────────────────────────────

def save_outputs(df: pd.DataFrame, root_causes: pd.DataFrame):
    print("\n" + "="*60)
    print("STEP 5: SAVING OUTPUTS")
    print("="*60)

    os.makedirs("output", exist_ok=True)

    output_cols = [
        "month","category",
        "budget_revenue","actual_revenue","total_variance","variance_pct",
        "volume_variance","price_variance","mix_variance","is_significant"
    ]
    df[output_cols].to_csv(OUT_VAR_CSV, index=False)
    print(f"  Variance decomposition → {OUT_VAR_CSV}")

    vol_total = df["volume_variance"].sum()
    prc_total = df["price_variance"].sum()
    mix_total = df["mix_variance"].sum()

    lines = [
        "ROOT CAUSE ANALYSIS REPORT",
        f"Generated : {datetime.now().strftime('%Y-%m-%d %H:%M')}",
        "="*55,
        "",
    ] + [f"  {f}" for f in findings] + [
        "",
        "VARIANCE DECOMPOSITION SUMMARY",
        f"  Volume Variance  : ₹{vol_total:,.0f}  "
        f"({'Favourable' if vol_total >= 0 else 'Adverse'})",
        f"  Price  Variance  : ₹{prc_total:,.0f}  "
        f"({'Favourable' if prc_total >= 0 else 'Adverse'})",
        f"  Mix    Variance  : ₹{mix_total:,.0f}  "
        f"({'Favourable' if mix_total >= 0 else 'Adverse'})",
        "",
        "METHODOLOGY",
        "  Volume variance = (Actual Vol - Budget Vol) × Budget Price",
        "  Price  variance = (Actual Price - Budget Price) × Actual Vol",
        "  Mix    variance = Total variance - Volume variance - Price variance",
        "",
        "ASSUMPTIONS",
        "  1. Significant threshold: >10% deviation from budget (configurable).",
        "  2. Mix variance is a residual — avoids double-counting with volume/price.",
        "  3. Missing actual data treated as zero-revenue (flagged separately).",
        "  4. All variances calculated at category-month level before aggregation.",
        "",
        "CHARTS",
        f"  {CHART_DIR}/trend_vs_budget.png",
        f"  {CHART_DIR}/category_contribution.png",
        f"  {CHART_DIR}/waterfall_variance.png",
    ]

    with open(OUT_REPORT, "w") as f:
        f.write("\n".join(lines))
    print(f"  Root cause report    → {OUT_REPORT}")


def main():
   

    df = load_data()
    df = decompose_variance(df)
    root_causes = identify_root_causes(df)
    visualise(df)
    save_outputs(df, root_causes)
    print("\nDone. Check output/ folder.")


if __name__ == "__main__":
    main()


STEP 1: LOAD DATA
  Merged: 48 rows | 12 months | 4 categories

STEP 2: PRICE-VOLUME DECOMPOSITION

  Total Budget Revenue : ₹16,429,376
  Total Actual Revenue : ₹16,236,567
  Total Variance       : ₹-192,809 (-1.2%)

STEP 3: ROOT CAUSE IDENTIFICATION

  Significant variances found: 9
  [2023-01] Haircare        ₹   -22,489 (-12.3%) — Primary driver: VOLUME shortfall (₹-15,105)
  [2023-03] Haircare        ₹    19,964 (+11.3%) — Primary driver: VOLUME uplift (₹18,725)
  [2023-04] Bath & Body     ₹    69,447 (+14.9%) — Primary driver: VOLUME uplift (₹49,051)
  [2023-07] Haircare        ₹  -189,927 (-30.4%) — Primary driver: VOLUME shortfall (₹-170,751)
  [2023-08] Haircare        ₹  -155,724 (-37.6%) — Primary driver: VOLUME shortfall (₹-132,128)
  [2023-09] Haircare        ₹  -183,780 (-32.1%) — Primary driver: VOLUME shortfall (₹-152,438)
  [2023-10] Skincare        ₹    97,352 (+18.0%) — Primary driver: VOLUME uplift (₹75,961)
  [2023-11] Skincare        ₹    75,548 (+18.7%) — Primar